In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pyspark.sql.functions as F

orders = spark.read.table('databricks_prep.data.orders')
display(orders)
orders.printSchema()
orders.limit(10).display()   


In [0]:
df=orders.select("customer_id","order_id","category","unit_price")
display(df)

In [0]:
df1 = df.filter(df.category == "Electronics")
display(df1)

In [0]:
df2 = df.filter(df.unit_price >750)
display(df)

In [0]:
df1 = orders.filter(orders.status == "Completed")
display(df1)

## Total SALES

In [0]:
orders1 = orders.withColumn("Sales",col("quantity")*col("unit_price"))
display(orders1)

### Sales by Each Category

In [0]:

orders = orders.withColumn("Sales",col("quantity")*col("unit_price"))
orders.groupBy("category").agg(sum("sales").alias("total_sales")).show()
display(orders)


### Average salary by Catagory

In [0]:
orders.groupBy("category").agg(avg("sales").alias("avg_sales")).show()

### Max salary in each catagory

In [0]:
orders.groupBy("category").agg(max("sales").alias("max_sales")).show()

### Find number of orders per customer

In [0]:
orders.groupBy("customer_id").count().show()

In [0]:
customer_df = spark.read.table("databricks_prep.data.customers")
display(customer_df)

### Join orders with customers

In [0]:
info = orders.join(customer_df,orders.customer_id == customer_df.customer_id,"inner")
info1 = info.select("order_id","customer_name","city","category","unit_price")
display(info)

### Find total sales by customer

In [0]:
sale = info.withColumn("Sales", info.unit_price * info.quantity)
sale1 = sale.groupBy("customer_name").agg(sum("Sales").alias("total_sales"))
display(sale1)

### Find customers who never placed an order

In [0]:
info = customer_df.join(orders,customer_df.customer_id == orders.customer_id,"left_anti")
display(info)

# Window Functions

## Load employees

In [0]:
employees = spark.read.table("databricks_prep.data.employees")
display(employees)

### Rank employees based on salary

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

In [0]:
windowSpec = Window.orderBy(col("salary").desc())
ranked_employees = employees.withColumn("rank", rank().over(Window.orderBy(col("salary").desc())))
display(ranked_employees)

### Find highest-paid employee in each department

without Rank

In [0]:
# Find max salary per department
max_salary_per_dept = employees.groupBy("department").agg(max("salary").alias("max_salary"))

# Join back to get employee details
high = employees.join(max_salary_per_dept, 
                      (employees.department == max_salary_per_dept.department) & 
                      (employees.salary == max_salary_per_dept.max_salary), 
                      "inner").select(employees["*"])
display(high)

with rank()

In [0]:
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

In [0]:
max_salary_per_dept = employees.withColumn("row_number",row_number().over(window_spec)).filter(col("row_number") == 1)
display(max_salary_per_dept)

###  Find top 2 employees from each department

In [0]:
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())
max_sal = employees.withColumn("row_number",row_number().over(window_spec)).filter(col("row_number")<=2)
display(max_sal)

In [0]:
%sql
select * from databricks_prep.data.employees

### Difference between employee salary and department average

In [0]:
avg_salary = employees.groupBy("department").agg(avg("salary").alias("avg_salary"))
avgd = employees.join(avg_salary,employees.department==avg_salary.department,"inner")
avg1 = avgd.withColumn("diff",col("salary")-col("avg_salary"))
display(avg1)



In [0]:
from pyspark.sql import functions as F
window_spec = Window.partitionBy("department")
df_resu = (employees.withColumn("dept_avg_salary",F.avg("salary").over(window_spec)).withColumn("salary_difference",F.col("salary")-F.col("dept_avg_salary")))
display(df_resu)
           

### Find second-highest salary in each department

In [0]:
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())
high = employees.withColumn("dense_rank",dense_rank().over(window_spec)).filter(col("dense_rank")==3)
display(high)

# Data Cleaning

### Find null values in transaction table

In [0]:
transactions = spark.read.table("databricks_prep.data.transactions")
display(transactions)

In [0]:
null_value = transactions.filter(col("payment_method").isNull()|col("city").isNull() |col("amount").isNull() ).show()
print(null_value)

### Replace null payment method with "Unknown"

In [0]:
from pyspark.sql.functions import coalesce, lit

df = transactions.withColumn(
    "payment_method",
    coalesce(col("payment_method"), lit("Unknown"))
)
display(df)

### Remove duplicate transactions

In [0]:
transactions.dropDuplicates(["transaction_id"]).show()

### Find duplicate transaction IDs

In [0]:
transactions.groupBy("transaction_id").count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
%sql
select * from databricks_prep.data.orders

### Extract year and month from order date

In [0]:
from pyspark.sql.functions import year, month

df_ord = orders.withColumn("year",year("order_date")).withColumn("month",month("order_date"))
df_ord1 = df_ord.select("year", "month")
display(df_ord1)


### Find monthly sales

In [0]:
display(df_ord)

In [0]:
monthly_sales = df_ord.groupBy("month").agg(sum("Sales").alias("total_sales"))
monthly_sales.show()

### Find orders from June 2024

In [0]:
sale_months = df_ord.filter(col("order_date").between("2024-06-01", "2025-12-31"))
display(sale_months)


### Calculate running sales by customer

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum

window_spec = Window.partitionBy("customer_id").orderBy("order_date")

df_running = df_ord.withColumn(
    "running_sales",
    sum("Sales").over(window_spec)
)

df_running.select(
    "customer_id",
    "order_date",
    "Sales",
    "running_sales"
).show()

### Find previous order amount

In [0]:
window_spec = Window.partitionBy("customer_id").orderBy("order_date")
df_previous = orders.withColumn(
    "previous_sales",
    lag("Sales", 1).over(window_spec)
)
display(df_previous)

### Find next order amount

In [0]:
window_spec = Window.partitionBy("customer_id").orderBy("order_date")
df_nxt = orders.withColumn(
    "nxt_sales",
    lead("Sales", 1).over(window_spec)
)
display(df_previous)

### Find customers whose current order is greater than their previous order

In [0]:
window_spec = Window.partitionBy("customer_id").orderBy("order_date")
df_prev = orders.withColumn(
    "prev_sales",
    lag("Sales").over(window_spec)
)
df_prev = df_prev.filter(col("Sales")>col("prev_sales"))
display(df_prev)

### Find top-selling category

In [0]:
top_cat = orders.groupBy("category").agg(sum("Sales").alias("total_sales"))
top_cat = top_cat.orderBy(col("total_sales").desc()).limit(1)
top_cat.show()

In [0]:
%sql
select * from databricks_prep.data.orders

### Top 3 customers by revenue

In [0]:
top_customers = orders.groupBy("customer_id").agg(sum("Sales").alias("total_sales"))
top_customers = top_customers.join(customer_df, "customer_id", "inner")
window_spec = Window.orderBy(col("total_sales").desc())
top_customers = top_customers.withColumn("rank", rank().over(window_spec))
top_customers = top_customers.filter(col("rank") <= 3)
display(top_customers)

### Top 2 products in each category

In [0]:
%sql
select * from databricks_prep.data.products;

In [0]:
products = spark.read.table("databricks_prep.data.products")

In [0]:
window_spec = Window.partitionBy("category").orderBy(col("price").desc())
df = products.withColumn("row_number", row_number().over(window_spec))
display(df)
df = df.filter(col("row_number") <=2)
display(df)

### Calculate net sales after discount

In [0]:

up_ord = orders.withColumn("net_sales",col("Sales")-col("discount"))
up_ord.select("Sales","discount","net_sales").show()


### Find category-wise revenue percentage

In [0]:
display(orders)

In [0]:
df2 = orders.groupBy("category").agg(sum("Sales").alias("category_revenue"))
total_revenue = orders.agg(sum("Sales").alias("total_revenue"))
result = df2.crossJoin(total_revenue).withColumn("revenue_percentage",col("category_revenue")/col("total_revenue"))
display(result)

### Find the highest order for every customer

In [0]:
window_spec = Window.partitionBy("customer_id").orderBy(col("Sales").desc())
high = orders.withColumn("row_num",row_number().over(window_spec))
display(high)
high2 = high.filter(col("row_num") == 1)
display(high)

### Find customers with more than 2 orders

In [0]:
high3 = high.filter(col("row_num") >= 2)
display(high3)


### Find the most frequently purchased category for each customer

In [0]:
from pyspark.sql.functions import count, row_number
from pyspark.sql.window import Window

# Step 1: Count purchases by customer and category
category_count = (
    orders
    .groupBy("customer_id", "category")
    .agg(count("*").alias("purchase_count"))
)
display(category_count)

# Step 2: Create window for each customer
window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy(category_count["purchase_count"].desc())
)

# Step 3: Rank categories
df_result = category_count.withColumn(
    "rank",
    row_number().over(window_spec)
)

# Step 4: Keep the most frequently purchased category
df_result = df_result.filter("rank = 1")

df_result.select(
    "customer_id",
    "category",
    "purchase_count"
).show()

### Find the percentage of completed, cancelled and returned orders

In [0]:
from pyspark.sql.functions import count as count_func, sum

status_count = orders.groupBy("status").agg(count_func('*').alias('count'))
total_count = status_count.agg(sum("count").alias("total")).collect()[0]["total"]
display(total_count)
status_count = status_count.withColumn("Percentage", (col("count") / total_count)*100)
display(status_count)

In [0]:
from pyspark.sql.functions import count
a = orders.groupBy("status").agg(count('*').alias('count'))
total = a.agg(sum("count").alias('total')).collect()[0]["total"]
b = a.withColumn("Perecentage",col("count")/total*100)
display(b)

### Create a Gold table

In [0]:
customer_orders = (
    orders
    .groupBy("customer_id")
    .agg(
        count("*").alias("total_orders"),

        sum(
            when(col("status") == "Completed", 1)
            .otherwise(0)
        ).alias("completed_orders"),

        sum(
            when(col("status") == "Completed", col("Sales"))
            .otherwise(0)
        ).alias("total_revenue"),

        avg(
            when(col("status") == "Completed", col("Sales"))
        ).alias("average_order_value"),

        max("order_date").alias("last_order_date")
    )
)

gold_customer = (
    customer_df
    .join(
        customer_orders,
        on="customer_id",
        how="left"
    )
)
gold_customer = gold_customer.select(
    "customer_id",
    "customer_name",
    "city",
    "state",
    "customer_segment",
    "total_orders",
    "completed_orders",
    "total_revenue",
    "average_order_value",
    "last_order_date"
)
display(gold_customer)

# ### Filtering + Columns

High-value orders

Find all orders where:

status = Completed
quantity >= 2
### unit_price > 500

In [0]:
comple = orders.filter((col("status") == "Completed") & (col("quantity")>=2) & (col("unit_price")>500))
display(comple)


In [0]:
comple.select("order_id","customer_id","category","quantity","unit_price").show()

### Q2. Create order value

Create a new column:

order_value = quantity * unit_price

Then find orders where order_value > 2000.

In [0]:
orders.filter(col("sales")>2000).display()

Apply discount

Create:

gross_amount
discount_amount
net_amount

Formula:

gross_amount = quantity * unit_price

net_amount = gross_amount - discount

In [0]:
a = orders.withColumn("gross_amount", col("quantity") * col("unit_price")) \
    .withColumn("discount_amount", col("discount")) \
    .withColumn("net_amount", col("gross_amount") - col("discount"))
display(a.select("gross_amount", "discount_amount", "net_amount"))